In [1]:
import os
import pandas as pd

In [2]:
# file paths
home = '/store/carroll/sbgplants/'
ref = os.path.join(home, 'schema')
raw = os.path.join(home, 'data', 'raw')

doi = os.path.join(raw, '10.15485.1618130') # Locations, metadata, and species cover from field sampling survey associated with NEON AOP survey, East River, CO 2018

out_folder = os.path.join(home, 'data', 'out_csv')

table = 'species_list'

In [3]:
# load schema and dtype
schema = pd.read_csv(os.path.join(ref, 'sbgplants-schema.csv'))
data_types = pd.read_csv(os.path.join(ref, 'data-types.csv'))

# view relevant schema
schema = schema[schema.table_name==table]
schema

,table_name,column_name,data_type
89,species_list,species_or_type,character
90,species_list,lifeform_code,integer
91,species_list,native_spp,boolean
92,species_list,species_genus_type,character


In [7]:
# no relevant data-types info

In [8]:
# no relevant output tables

In [5]:
# load relevant raw tables
fractional_cover = pd.read_csv(os.path.join(doi, 'fractional_cover.csv'))

species_list = pd.read_csv(os.path.join(doi, 'species_list.csv'))
species_list = species_list.dropna(axis=1, how='all').dropna(axis=0, how='all') # drop all na rows and columns
species_list

,CoverCode,Family,Genus,Species,AltFieldCode,Notes
0,AcoRos,Rosaceae,Acomastylis,rossii,GeuRos,*syn. = Geum rossii
1,AgaUrt,Lamiaceae,Agastache,urticifolia,NaN,NaN
2,AgrSpp,Poaceae,Agrostis,spp,NaN,NaN
3,Alder,Betulaceae,Alnus,incana,NaN,NaN
4,AneMul,Ranunculaceae,Anemone,multifida,NaN,NaN
...,...,...,...,...,...,...
102,VicAme,Fabaceae,Vicia,americana,NaN,NaN
103,Willow,Salicaceae,Salix,spp,NaN,NaN
104,wolfii,Salicaceae,Salix,wolfii,NaN,NaN
105,WyeAmp,Asteraceae,Wyethia,amplexicaulis,NaN,NaN


In [6]:
# prepare & populate out table
out_table = pd.DataFrame(columns=schema['column_name'].unique())

out_table['species_or_type'] = species_list['Genus'] + ' ' + species_list['Species']
out_table.loc[out_table['species_or_type'].isna(), 'species_or_type'] = species_list.loc[species_list['Genus'].isna(), 'CoverCode']
out_table['species_genus_type'] = species_list['Genus'].str[:4].str.upper() + species_list['Species'].str[:4].str.upper()

# out_table['lifeform_code'] = ? # NA for now, still determining system 
# out_table['native_spp'] = ? # NA for now... unclear how to define

out_table

,species_or_type,lifeform_code,native_spp,species_genus_type
0,Acomastylis rossii,NaN,NaN,ACOMROSS
1,Agastache urticifolia,NaN,NaN,AGASURTI
2,Agrostis spp,NaN,NaN,AGROSPP
3,Alnus incana,NaN,NaN,ALNUINCA
4,Anemone multifida,NaN,NaN,ANEMMULT
...,...,...,...,...
102,Vicia americana,NaN,NaN,VICIAMER
103,Salix spp,NaN,NaN,SALISPP
104,Salix wolfii,NaN,NaN,SALIWOLF
105,Wyethia amplexicaulis,NaN,NaN,WYETAMPL


In [7]:
# confirm final column data types

print(out_table.dtypes)

# # adjust as necessary
# out_table['lifeform_code'] = out_table['lifeform_code'].astype(int)
# out_table['native_spp'] = out_table['native_spp'].astype(bool)

out_table.dtypes

species_or_type       object
lifeform_code         object
native_spp            object
species_genus_type    object
dtype: object


species_or_type       object
lifeform_code         object
native_spp            object
species_genus_type    object
dtype: object

In [8]:
# check unique values for spp list, frac cover

sorted(list(set(out_table['species_or_type'])))

['Abies lasiocarpa',
 'Acomastylis rossii',
 'Agastache urticifolia',
 'Agrostis spp',
 'Alnus incana',
 'Anemonastrum narcissiflorum ssp. Zephyrum',
 'Anemone multifida',
 'Aquilegia coerulea var. coerulea',
 'Arnica mollis',
 'Arnica parryi',
 'Artemisia dracunculus',
 'Artemisia tridentata',
 'Bare',
 'Betula glandulosa',
 'Bistorta bistortoides',
 'Bromopsis inermis',
 'Calamagrostis stricta',
 'Carex aquatilis',
 'Carex hoodii',
 'Carex lenticularis',
 'Carex microptera',
 'Carex siccata',
 'Carex spp',
 'Carex utriculata',
 'Castilleja rhexiifolia',
 'Castilleja sulphurea',
 'Chamerion danielsii',
 'Clementsia rhodantha',
 'Corydalis caseana',
 'Delphinium barbeyi',
 'Deschampsia cespitosa',
 'Distegia involucrata',
 'Dugaldia hoopesii',
 'Elymus lanceolatus',
 'Elymus spp',
 'Erigeron glacialis',
 'Erigeron speciosus',
 'Erythronium grandiflorum',
 'Eucephalus engelmannii',
 'Festuca idahoensis',
 'Festuca spp',
 'Festuca thurberi',
 'Fragaria virgiana',
 'Frasera speciosa',
 'G

In [9]:
# export table
fp_out = os.path.join(out_folder, f'{table}.csv')
out_table.to_csv(fp_out, index=False)